In [2]:
import numpy as np
import os
from os import path, listdir
import pickle

# large3 = np.load("data_npz/large_tight_bound.npz")
# print(large3.files)
# raw = large3["raw"]
# print(raw.shape)
# print(raw)
# print(large3["priors_samples"].shape)


In [3]:
sweep_results_dir = "sweep_results"

files= listdir(sweep_results_dir)

for file in files:
    if file.endswith(".pkl"):
        with open(path.join(sweep_results_dir, file), "rb") as f:
            data = pickle.load(f)
            
        print(f"File: {file}")
        print(f"Data keys: {data.keys()}")
        for key, value in data.items():
            if key == "lambda_p": continue
            sweep_vals = value["sweep_values"]
            median_means = value["median_means"]
            lower_bounds = value["lower_bounds"]
            upper_bounds = value["upper_bounds"]
            cov_90 = value["coverage_90"]
        
            print(f"  {key}:")
            print(f"    Sweep values: {sweep_vals}")
            print(f"    Median means: {median_means}")
            print(f"    90% CI lower bounds: {lower_bounds}")
            print(f"    90% CI upper bounds: {upper_bounds}")
            print(f"    90% CI coverage: {cov_90}")

File: sweep_posterior_medians_stat_m,s,min_gr,max_gr,ac1,ac2,ac3,q25_gr,q50_gr,q75_gr_from_posterior_1000_20260622_114614.png.pkl
Data keys: dict_keys(['omega', 'lambda_p', 'pi_bar'])
  omega:
    Sweep values: [0.125 0.25  0.375 0.5   0.625 0.75  0.875]
    Median means: [0.48207942 0.48432758 0.48400736 0.48725566 0.4885523  0.48904088
 0.48951593]
    90% CI lower bounds: [0.41216332 0.41226292 0.41006359 0.41458642 0.41645339 0.41620818
 0.4181143 ]
    90% CI upper bounds: [0.55675524 0.55917847 0.56051034 0.5601868  0.56481755 0.56697357
 0.56210285]
    90% CI coverage: [0.97799999 1.         1.         1.         1.         1.
 0.99000001]
  pi_bar:
    Sweep values: [0.1875 0.375  0.5625 0.75   0.9375 1.125  1.3125]
    Median means: [0.60918128 0.64762622 0.68953484 0.75171018 0.78721941 0.83462006
 0.85557848]
    90% CI lower bounds: [0.26399422 0.28501469 0.34523997 0.41021097 0.45168188 0.50319451
 0.54124945]
    90% CI upper bounds: [0.97514141 0.98266739 1.01034331 1.0

In [11]:
recovery_results_dir = "cov_results"

files = listdir(recovery_results_dir)
for file in files:
    if file.endswith(".pkl"):
        with open(path.join(recovery_results_dir, file), "rb") as f:
            data = pickle.load(f)
            
        print(f"File: {file}")
        print(f"Data keys: {data.keys()}")
        
        coverage_results = data["coverage_results"]
        mean_median = data["mean_median"]
        std_median = data["std_median"]
        mean_interval_width = data["mean_interval_width"][0]
        levels = data["levels"]

        if "medians" in data.keys():
            median = data["medians"]
            median_median = np.median(median, axis=0)
            median_median_idx = np.argmin(np.abs(median - median_median), axis=0)
            q1 = np.quantile(median, 0.025, axis=0)
            q3 = np.quantile(median, 0.975, axis=0)

            print(f"    Median of medians: {' & '.join(f'{m:.2f} ({l:.2f}-{u:.2f})' for m, l, u in zip(median_median, q1, q3))}")


        medians_std = zip(mean_median, std_median)
        print(coverage_results.keys())
        coverage_results = coverage_results[f"{(1 - 2 * np.float32(0.025)):3.0%}"]

        print(f"    Coverage results: {' | '.join(f'{x:2.2%}' for x in coverage_results)}")
        print(f"    Mean of medians: {' & '.join(f'{mean:.3f} ({cov:.3f})' for mean, cov in zip(mean_median, coverage_results))}")
        print(f"    Std of medians: {np.array2string(std_median, precision=3, suppress_small=False)}")
        print(f"    Mean interval width: {' & '.join(f'{width:.3f}' for width in mean_interval_width)}")
        print(f"    Levels: {np.array2string(levels, precision=3, suppress_small=False)}")


File: coverage_nn_simple_hierarchical_20260622_162726.pkl
Data keys: dict_keys(['coverage_results', 'mean_median', 'std_median', 'mean_interval_width', 'true', 'levels'])
dict_keys(['95%', '90%', '80%', '50%'])
    Coverage results: 99.90% | 100.00% | 92.60%
    Mean of medians: 0.442 (0.999) & 2.536 (1.000) & 0.984 (0.926)
    Std of medians: [0.166 0.095 0.107]
    Mean interval width: 0.872 & 4.731 & 0.385
    Levels: [0.025 0.05  0.1   0.25 ]
File: coverage_statistics_m,s,min_gr,max_gr,ac1,ac2,ac3,q25_gr,q50_gr,q75_gr_20260624_103955.pkl
Data keys: dict_keys(['coverage_results', 'mean_median', 'std_median', 'medians', 'mean_interval_width', 'true', 'levels'])
    Median of medians: 0.48 (0.40-0.56) & 2.53 (2.24-2.78) & 0.81 (0.37-1.10)
dict_keys(['95%', '90%', '80%', '50%'])
    Coverage results: 100.00% | 100.00% | 98.80%
    Mean of medians: 0.481 (1.000) & 2.521 (1.000) & 0.788 (0.988)
    Std of medians: [0.041 0.138 0.195]
    Mean interval width: 0.903 & 4.533 & 1.269
    Lev

In [5]:
from sbi.examples.minimal import simple
from sbi.diagnostics import run_sbc, check_sbc
from sbi.utils import BoxUniform

import torch

param_dim = 3
reduce_fns = [eval(f"lambda theta, x: theta[:, {i}]") for i in range(param_dim)]

x_o = torch.ones(3)

prior = BoxUniform(low=torch.ones(3) * (-2), high=torch.ones(3) * 2)

post = simple()
post.set_default_x(x_o)

reduce_fns.append(lambda theta, x: -post.log_prob(theta, x))

theta = prior.sample((200,))

xs = torch.randn(200, 3)

ranks, dap_samples, posterior_samples = run_sbc(theta, xs, post, reduce_fns=reduce_fns)
print(ranks.shape)
print(dap_samples.shape)
print(posterior_samples.shape)


/Users/jerrze/Projects/Thesis/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 500/500 [00:00<00:00, 57972.41it/s]


 Neural network successfully converged after 126 epochs.

100%|██████████| 100/100 [00:00<00:00, 4091.96it/s]
/Users/jerrze/Projects/Thesis/venv/lib/python3.14/site-packages/sbi/utils/diagnostics_utils.py:45: UserWarning: Capping max_sampling_batch_size from 10000 to 500 to avoid excessive memory usage.
  posterior_samples = posterior.sample_batched(
Calculating ranks for 200 SBC samples: 100%|██████████| 200/200 [01:29<00:00,  2.23it/s]

torch.Size([200, 4])
torch.Size([200, 3])
torch.Size([1000, 200, 3])


In [6]:
sbc_res_dir = "sbc_res"

files = listdir(sbc_res_dir)
for file in files:
    if file.endswith(".pkl"):
        with open(path.join(sbc_res_dir, file), "rb") as f:
            data = pickle.load(f)
            
        print(f"File: {file}")
        print(f"Data keys: {data.keys()}")
        
        for key, value in data.items():
            print(value.keys())
            print(f"  {key}:")
            print(f"    Ranks shape: {value['ranks'].shape}")
            print(f"    DAP samples shape: {value['dap_samples'].shape}")

            cov_res95 = value["coverage"][0, :] ### 95%
            cov_res90 = value["coverage"][1, :] ### 90%

            mean_rank = value["ranks"].float().mean(dim=0)
            std_rank = value["ranks"].float().std(dim=0)
            print(f"    Mean ranks: {" & ".join(f"{mean:.2f} ({std:.2f})" for mean, std in zip(mean_rank, std_rank))}")

            ### rank appx discrete uniform 0, 1000 -> var (r) = (N + 1) (N - 1) / 12
            sd_r = ((1001 + 1) * (1001 - 1) / 12) ** 0.5
            sde_r = sd_r / (1000 ** 0.5)
            z_scores = (mean_rank - 500) / sde_r
            print(f"    Z-scores: {' & '.join(f'{z:.2f}' for z in z_scores)}")

            print(f"    Coverage (95%): {' & '.join(f'{c1:.3f}' for c1, c2 in zip(cov_res95, cov_res90))}")
            print(f"    Coverage (90%): {' & '.join(f'{c2:.3f} | {c1:.3f}' for c1, c2 in zip(cov_res95, cov_res90))}")

File: sbc_results_IT_prior_1000_post_1000_20260622_170518_seed_0.pkl
Data keys: dict_keys(['npe_simple_hierarchical', 'npe_hierarchical', 'npe_seq_multivariate', 'npe_m,s,min_gr,max_gr,ac1,ac2,ac3,q25_gr,q50_gr,q75_gr'])
dict_keys(['ks_pvals', 'c2st_ranks', 'c2st_dap', 'ranks', 'dap_samples', 'coverage'])
  npe_simple_hierarchical:
    Ranks shape: torch.Size([1000, 3])
    DAP samples shape: torch.Size([1000, 3])
    Mean ranks: 531.15 (298.20) & 501.33 (288.65) & 516.88 (315.10)
    Z-scores: 3.41 & 0.15 & 1.85
    Coverage (95%): 0.930 & 0.943 & 0.921
    Coverage (90%): 0.882 | 0.930 & 0.906 | 0.943 & 0.846 | 0.921
dict_keys(['ks_pvals', 'c2st_ranks', 'c2st_dap', 'ranks', 'dap_samples', 'coverage'])
  npe_hierarchical:
    Ranks shape: torch.Size([1000, 3])
    DAP samples shape: torch.Size([1000, 3])
    Mean ranks: 490.69 (295.57) & 506.24 (293.10) & 541.99 (292.74)
    Z-scores: -1.02 & 0.68 & 4.59
    Coverage (95%): 0.943 & 0.936 & 0.954
    Coverage (90%): 0.890 | 0.943 & 0.8